In [1]:
import pandas as pd
import scanpy as sc
import os
import anndata

In [2]:
os.chdir("/home/lixiangyu/zr/Annotate/")

##### output data:/home/lixiangyu/zr/Annotate/output_data/public_data/Mouse_all/Mouse_merged.h5ad

# AS

## GSE239591 29839 27094

In [3]:
file_paths = [
    "data/Mouse_public_data/GSE239591/GSE239591_ABT_f.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_ABT_M.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_Ctrl_F.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_Ctrl_M.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_HFD_F.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_HFD_M.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
]

In [4]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)


Number of cells in sample Sample1: 6084
Number of cells in sample Sample2: 4888
Number of cells in sample Sample3: 5469
Number of cells in sample Sample4: 5775
Number of cells in sample Sample5: 2635
Number of cells in sample Sample6: 4988


In [5]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

adata_final.write_h5ad("data/Mouse_public_data/GSE239591/GSE239591_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE239591/GSE239591_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
adata_final

AnnData object with n_obs × n_vars = 29839 × 32288
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [8]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [9]:
mt_genes_lower

['mt-Nd1',
 'mt-Nd2',
 'mt-Co1',
 'mt-Co2',
 'mt-Atp8',
 'mt-Atp6',
 'mt-Co3',
 'mt-Nd3',
 'mt-Nd4l',
 'mt-Nd4',
 'mt-Nd5',
 'mt-Nd6',
 'mt-Cytb']

In [10]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [11]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGAATCGCG-1               1388        3271.0       2.170590
AAACCCAAGACCTCCG-1               1995       10631.0       0.338632
AAACCCAAGAGATCGC-1               1506        4098.0       3.562713
AAACCCAAGGTAAAGG-1               4649       18316.0       1.676130
AAACCCACAGACGCTC-1               2017        5531.0       3.869102

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [12]:
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [13]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 18] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 29839


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 29036
Number of cells beforeMT filter: 28061
Number of cells after MT filter: 27094


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [14]:
adata_final.write("data/Mouse_public_data/GSE239591/GSE239591_postQC.h5ad")

In [4]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE239591/GSE239591_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
adata_final.obs['sample'].value_counts()

sample
GSE239591_1    5323
GSE239591_4    5319
GSE239591_3    5081
GSE239591_2    4514
GSE239591_6    4472
GSE239591_5    2385
Name: count, dtype: int64

## GSE269449 12370 10158

In [15]:

file_paths = [
    "data/Mouse_public_data/GSE269449/GSE269449_LXR.h5ad",
    "data/Mouse_public_data/GSE269449/GSE269449_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 7878
Number of cells in sample Sample2: 4492


In [16]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE269449/GSE269449_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [17]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [18]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE269449/GSE269449_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [19]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGCCTATTG-1               2236        6313.0       1.457310
AAACCCAAGGGTTGCA-1                296         908.0       2.643172
AAACCCACAAAGCGTG-1               1699        3611.0       3.461645
AAACCCACACGACAGA-1               3252        9508.0       2.166597
AAACCCACACGACCTG-1               6047       27013.0       1.003221

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [20]:
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [21]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 12370


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 12288
Number of cells beforeMT filter: 11746
Number of cells after MT filter: 10158


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [22]:
adata_final.write("data/Mouse_public_data/GSE269449/GSE269449_postQC.h5ad")

In [7]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE269449/GSE269449_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [8]:
adata_final.obs['sample'].value_counts()

sample
GSE269449_2    6565
GSE269449_1    3593
Name: count, dtype: int64

## GSE298574 28892 26239

In [23]:
file_paths = [
    "data/Mouse_public_data/GSE298574/GSE298574_KO.h5ad",
    "data/Mouse_public_data/GSE298574/GSE298574_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 7766
Number of cells in sample Sample2: 21126


In [24]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE298574/GSE298574_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [25]:
adata_final

AnnData object with n_obs × n_vars = 28892 × 33696
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [26]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 255 个包含 'mt' 的基因


In [27]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE298574/GSE298574_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [28]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGTATGCAA-1               5862       27665.0       4.370143
AAACCCAAGTTGTAAG-1               1057        2145.0      13.986013
AAACCCACAAAGAGTT-1               2072        5116.0      13.193901
AAACCCACACGGTCTG-1               4961       22302.0       9.339970
AAACCCAGTACCTAAC-1               2185        6663.0       2.416329

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [29]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 18] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 28892


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 28255
Number of cells beforeMT filter: 27807
Number of cells after MT filter: 26239


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [30]:
adata_final.write("data/Mouse_public_data/GSE298574/GSE298574_postQC.h5ad")

In [9]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE298574/GSE298574_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [10]:
adata_final.obs['sample'].value_counts()

sample
GSE298574_2    19413
GSE298574_1     6826
Name: count, dtype: int64

## GSE274572 12258 10515

In [31]:
file_paths = [
    "data/Mouse_public_data/GSE274572/GSE274572_D11.h5ad",
    "data/Mouse_public_data/GSE274572/GSE274572_NS.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 6234
Number of cells in sample Sample2: 6024


In [32]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE274572/GSE274572_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [33]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 279 个包含 'mt' 的基因


In [34]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE274572/GSE274572_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [35]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGCGTCGAA-1                740        1236.0       5.987055
AAACCCAAGTCTGGTT-1               4629       20502.0       0.858453
AAACCCACACCTTCGT-1               2345        6596.0       2.152820
AAACCCACATTCACCC-1               1978        4243.0       4.996465
AAACCCAGTGTGATGG-1               1211        2600.0       8.000000

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [36]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=500)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 7.5] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 12258


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 12074
Number of cells beforeMT filter: 11854
Number of cells after MT filter: 10515


In [37]:
adata_final.write("data/Mouse_public_data/GSE274572/GSE274572_postQC.h5ad")

In [11]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE274572/GSE274572_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [12]:
adata_final.obs['sample'].value_counts()

sample
GSE274572_2    5535
GSE274572_1    4980
Name: count, dtype: int64

## GSE210406 18545 15843

In [38]:
file_paths = [
    "data/Mouse_public_data/GSE210406/GSE210406_TRF2_HFD.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_TRF2_TA.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_WT_HFD.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 2358
Number of cells in sample Sample2: 6257
Number of cells in sample Sample3: 4250
Number of cells in sample Sample4: 5680


In [39]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE210406/GSE210406_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [40]:
adata_final

AnnData object with n_obs × n_vars = 18545 × 55492
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [41]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 0 个以 'mt-' 开头的基因
找到 0 个包含 'mt' 的基因


In [42]:
adata_final.var_names

Index(['ENSMUSG00000102693', 'ENSMUSG00000064842', 'ENSMUSG00000051951',
       'ENSMUSG00000102851', 'ENSMUSG00000103377', 'ENSMUSG00000104017',
       'ENSMUSG00000103025', 'ENSMUSG00000089699', 'ENSMUSG00000103201',
       'ENSMUSG00000103147',
       ...
       'ENSMUSG00000094431', 'ENSMUSG00000094621', 'ENSMUSG00000098647',
       'ENSMUSG00000096730', 'ENSMUSG00000095742', 'mCereulean_cDNA',
       'hrGFP_cDNA', 'EYFP_cDNA', 'tdimer2_12_cDNA', 'hTRF2_transgene'],
      dtype='object', name='gene_ids', length=55492)

In [43]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE210406/GSE210406_merged.h5ad")
ensembl_id_df = pd.read_csv("mouse_gene_names_to_ensembl.csv")
# 核心修改：创建“基因ID→基因名”的映射字典（原代码是“基因名→基因ID”）
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
# 备份原始的基因ID
adata_final.var['original_ensembl_ids'] = adata_final.var_names

# 将AnnData的变量名（基因ID）转换为基因名
# 逻辑：如果基因ID在映射字典中，则替换为对应的基因名；否则保留原始ID
adata_final.var_names = [
    ensembl_to_gene[ensembl_id] if ensembl_id in ensembl_to_gene else ensembl_id 
    for ensembl_id in adata_final.var_names
]

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [44]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 299 个包含 'mt' 的基因


In [45]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [46]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAGTATTCCGA-1               1256        2823.0       2.798441
AAACCCATCGCCGTGA-1               1373        2439.0       1.517015
AAACGAAGTATTGGCT-1               4152       13688.0       0.694039
AAACGAAGTTAAGGGC-1               1354        2735.0       7.166362
AAACGCTAGGAATTAC-1               2014        4391.0       0.204965

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [47]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 18545


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 18472
Number of cells beforeMT filter: 17472
Number of cells after MT filter: 15843


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [48]:
adata_final.write("data/Mouse_public_data/GSE210406/GSE210406_postQC.h5ad")

In [13]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE210406/GSE210406_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [14]:
adata_final.obs['sample'].value_counts()

sample
GSE210406_2    6141
GSE210406_3    5621
GSE210406_4    2507
GSE210406_1    1574
Name: count, dtype: int64

## GSE262694 6845804 86091

In [234]:
file_paths = [
    "data/Mouse_public_data/GSE262694/GSE262694_KO_12wks.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_12wks_rep1.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_12wks_rep2.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_30wks_rep1.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_30wks_rep2.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 9737
Number of cells in sample Sample2: 9379
Number of cells in sample Sample3: 11720
Number of cells in sample Sample4: 20088
Number of cells in sample Sample5: 6794880


In [235]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE262694/GSE262694_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [236]:
adata_final

AnnData object with n_obs × n_vars = 6845804 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [237]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [242]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE262694/GSE262694_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [243]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCACAACTCCAA-1               5289       21333.0       3.698495
AAACCCACACCTGCGA-1               3188       11756.0       2.152092
AAACCCACACGCAAAG-1               2924       15642.0       1.726122
AAACCCACACGCGTCA-1               4132       16640.0       3.804086
AAACCCACAGGACATG-1               1902        5635.0       3.229814

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [244]:
adata_final.obs_names_make_unique()
adata_final.var_names_make_unique()

In [245]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 200000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 6845804
Number of cells before counts filter: 89101
Number of cells beforeMT filter: 89101
Number of cells after MT filter: 86091


In [246]:
adata_final.write("data/Mouse_public_data/GSE262694/GSE262694_postQC.h5ad")

In [15]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE262694/GSE262694_postQC.h5ad")

In [16]:
adata_final.obs['sample'].value_counts()

sample
GSE262694_5    37835
GSE262694_4    20061
GSE262694_3    10863
GSE262694_1     8852
GSE262694_2     8480
Name: count, dtype: int64

## GSE284253 5475 4961

In [19]:
file_paths = [
    "data/Mouse_public_data/GSE284253/GSE284253_AAA.h5ad",
    "data/Mouse_public_data/GSE284253/GSE284253_Naive.h5ad",
    "data/Mouse_public_data/GSE284253/GSE284253_RA.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 2842
Number of cells in sample Sample2: 782
Number of cells in sample Sample3: 1851


In [20]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE284253/GSE284253_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [21]:
adata_final

AnnData object with n_obs × n_vars = 5475 × 27998
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [22]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 231 个包含 'mt' 的基因


In [23]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE284253/GSE284253_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [24]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGCGATAGC-1               3471       13631.0       2.376935
AAACCTGAGTGGCACA-1               3147       12301.0       1.796602
AAACCTGCATTAACCG-1               1382        4020.0       3.482587
AAACCTGGTCCAGTGC-1               1446        3774.0       1.854796
AAACCTGGTGGTCCGT-1                892        2073.0       4.823927

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [25]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=3000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 7.5] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 5475
Number of cells before counts filter: 4981
Number of cells beforeMT filter: 4981
Number of cells after MT filter: 4961


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [26]:
adata_final.write("data/Mouse_public_data/GSE284253/GSE284253_postQC.h5ad")

In [27]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE284253/GSE284253_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [28]:
adata_final.obs['sample'].value_counts()

sample
GSE284253_1    2533
GSE284253_3    1656
GSE284253_2     772
Name: count, dtype: int64

## GSE279601 2182871 15679

In [67]:
file_paths = [
    "data/Mouse_public_data/GSE279601/GSE279601_rep1_MT.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_rep1_WT.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_rep2_MT.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_rep2_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 487632
Number of cells in sample Sample2: 509697
Number of cells in sample Sample3: 610398
Number of cells in sample Sample4: 575144


In [68]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE279601/GSE279601_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [69]:
adata_final

AnnData object with n_obs × n_vars = 2182871 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [70]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [71]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE279601/GSE279601_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [72]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGAAACCCA-1                  0           0.0            NaN
AAACCCAAGAAAGAAC-1                  1           1.0            0.0
AAACCCAAGAAAGCGA-1                  2           2.0            0.0
AAACCCAAGAACTACG-1                  1           1.0            0.0
AAACCCAAGAAGCTGC-1                 22          28.0            0.0

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [73]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=6500) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 2182871


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 16445
Number of cells beforeMT filter: 16296
Number of cells after MT filter: 15679


In [74]:
adata_final.write("data/Mouse_public_data/GSE279601/GSE279601_postQC.h5ad")

In [29]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE279601/GSE279601_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [30]:
adata_final.obs['sample'].value_counts()

sample
GSE279601_1    5521
GSE279601_2    3979
GSE279601_3    3593
GSE279601_4    2586
Name: count, dtype: int64

## GSE211216 7673 6857

In [75]:
file_paths = [
    "data/Mouse_public_data/GSE211216/GSE211216_Control.h5ad",
    "data/Mouse_public_data/GSE211216/GSE211216_Diabetes.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    adata_sample.var_names_make_unique()
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample1: 4215
Number of cells in sample Sample2: 3458


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [76]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE211216/GSE211216_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [77]:
adata_final

AnnData object with n_obs × n_vars = 7673 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [78]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [79]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE211216/GSE211216_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [80]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGAGACTTA-1               1170        2175.0       4.183908
AAACCTGAGAGTGAGA-1               2019        7244.0       1.697957
AAACCTGAGATCCGAG-1               1367        3298.0       1.425106
AAACCTGAGCTTATCG-1               1858        5629.0       4.707763
AAACCTGAGTCACGCC-1                787        1603.0       2.557704

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [81]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=20000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 7673
Number of cells before counts filter: 7180
Number of cells beforeMT filter: 7142
Number of cells after MT filter: 6857


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [82]:
adata_final.write("data/Mouse_public_data/GSE211216/GSE211216_postQC.h5ad")

In [31]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE211216/GSE211216_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [33]:
adata_final.obs['sample'].value_counts()

sample
GSE211216_1    4036
GSE211216_2    2821
Name: count, dtype: int64

## GSE245373 5979 5821

In [83]:
file_paths = [
    "data/Mouse_public_data/GSE245373/GSE245373_advAp.h5ad",
    "data/Mouse_public_data/GSE245373/GSE245373_advWT.h5ad",
    "data/Mouse_public_data/GSE245373/GSE245373_bAPOE.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 3153
Number of cells in sample Sample2: 2271
Number of cells in sample Sample3: 555


In [84]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE245373/GSE245373_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [85]:
adata_final

AnnData object with n_obs × n_vars = 5979 × 55358
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [86]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 13 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 462 个包含 'mt' 的基因


In [87]:
mt_genes_upper

['MT-ATP6',
 'MT-ATP8',
 'MT-CO1',
 'MT-CO2',
 'MT-CO3',
 'MT-CYTB',
 'MT-ND1',
 'MT-ND2',
 'MT-ND3',
 'MT-ND4',
 'MT-ND4L',
 'MT-ND5',
 'MT-ND6']

In [88]:
mt_genes_lower

['mt-Atp6',
 'mt-Atp8',
 'mt-Co1',
 'mt-Co2',
 'mt-Co3',
 'mt-Cytb',
 'mt-Nd1',
 'mt-Nd2',
 'mt-Nd3',
 'mt-Nd4',
 'mt-Nd4l',
 'mt-Nd5',
 'mt-Nd6']

In [89]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE245373/GSE245373_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [90]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE245373/GSE245373_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [91]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGCTAACTC-1                725        2256.0            0.0
AAACCTGCAATTCCTT-1               2473        7937.0            0.0
AAACCTGCACACCGAC-1               1451        9171.0            0.0
AAACCTGCATAGGATA-1                861        1744.0            0.0
AAACCTGCATCAGTCA-1                898        2253.0            0.0

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Atp6', 'mt-Atp8', 'mt-Co1', 'mt-Co2', 'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3', 'mt-Nd4', 'mt-Nd4l', 'mt-Nd5', 'mt-Nd6']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [92]:
print(adata_final.obs['pct_counts_mt'].value_counts())

pct_counts_mt
0.000000    3708
1.785714       3
1.550388       3
1.666667       2
1.574803       2
            ... 
1.403785       1
1.311189       1
0.732104       1
1.522567       1
1.057318       1
Name: count, Length: 2239, dtype: int64


In [93]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 200000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 3] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 5979
Number of cells before counts filter: 5972
Number of cells beforeMT filter: 5972
Number of cells after MT filter: 5821


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [94]:
adata_final.write("data/Mouse_public_data/GSE245373/GSE245373_postQC_mt.h5ad")

In [34]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE245373/GSE245373_postQC_mt.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [35]:
adata_final.obs['sample'].value_counts()

sample
GSE245373_1    3150
GSE245373_2    2117
GSE245373_3     554
Name: count, dtype: int64

## GSE264253 11102 9915

In [95]:
file_paths = [
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_CD31.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_CD45.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_Live.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_YFP.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_CD31.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_CD45.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_Live.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_YFP.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 295
Number of cells in sample Sample2: 1621
Number of cells in sample Sample3: 505
Number of cells in sample Sample4: 5789
Number of cells in sample Sample5: 80
Number of cells in sample Sample6: 560
Number of cells in sample Sample7: 404
Number of cells in sample Sample8: 1848


In [96]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE264253/GSE264253_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [97]:
adata_final

AnnData object with n_obs × n_vars = 11102 × 28693
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [98]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 231 个包含 'mt' 的基因


In [99]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE264253/GSE264253_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [100]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACGGGGTTACTGAC-1                237         562.0      37.366547
AAACGGGTCCACTCCA-1               2420        7621.0       2.256922
AAAGCAAAGGGCATGT-1               4639       21825.0       0.902635
AAAGTAGAGAGGTACC-1               4919       32268.0       1.050576
AAATGCCTCCGTTGTC-1               3298       11593.0       0.810834

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [101]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc 100
sc.pp.filter_cells(adata_final, max_genes=5000) # 10000
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 11102
Number of cells before counts filter: 10776
Number of cells beforeMT filter: 10323
Number of cells after MT filter: 10099


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [102]:
adata_final.write("data/Mouse_public_data/GSE264253/GSE264253_postQC.h5ad")

In [36]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE264253/GSE264253_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [37]:
adata_final.obs['sample'].value_counts()

sample
GSE264253_4    5561
GSE264253_8    1598
GSE264253_2    1376
GSE264253_6     556
GSE264253_3     434
GSE264253_7     313
GSE264253_1     190
GSE264253_5      71
Name: count, dtype: int64

## GSE209525 16820 15387

In [103]:
file_paths = [
    "data/Mouse_public_data/GSE209525/GSE209525_1_ND.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_2_ND.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_3_HFD.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_4_HFD.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_5_HFD_VG.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_6_HFD_VG.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 3399
Number of cells in sample Sample2: 1577
Number of cells in sample Sample3: 4055
Number of cells in sample Sample4: 1449
Number of cells in sample Sample5: 2725
Number of cells in sample Sample6: 3615


In [104]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE209525/GSE209525_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [105]:
adata_final

AnnData object with n_obs × n_vars = 16820 × 32288
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [106]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [107]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE209525/GSE209525_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [108]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGGTAGATT-1               1262        2316.0       1.424870
AAACCCACATAAGCAA-1               2221        7483.0       2.138180
AAACCCACATACATCG-1               4553       17310.0       2.403235
AAACCCAGTGGTCTTA-1               2200        4852.0       1.957955
AAACCCATCAAATGCC-1               4910       25370.0       5.514387

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [109]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 16820


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 16673
Number of cells beforeMT filter: 16474
Number of cells after MT filter: 15387


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [110]:
adata_final.write("data/Mouse_public_data/GSE209525/GSE209525_postQC.h5ad")

In [38]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE209525/GSE209525_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [39]:
adata_final.obs['sample'].value_counts()

sample
GSE209525_3    3483
GSE209525_6    3439
GSE209525_1    3246
GSE209525_5    2479
GSE209525_2    1475
GSE209525_4    1265
Name: count, dtype: int64

## GSE205930 40208 38460

In [111]:
file_paths = [
    "data/Mouse_public_data/GSE205930/GSE205930_C_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_ED_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_IC_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_LD_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_PL_AO.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 10704
Number of cells in sample Sample2: 7019
Number of cells in sample Sample3: 7756
Number of cells in sample Sample4: 9030
Number of cells in sample Sample5: 5699


In [112]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE205930/GSE205930_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [113]:
adata_final

AnnData object with n_obs × n_vars = 40208 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [114]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [115]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE205930/GSE205930_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [116]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGCAGATCG-1               1045        2167.0       2.768805
AAACCTGAGCGATTCT-1               1006        2182.0      11.274060
AAACCTGAGTTGAGTA-1               1886        4501.0       6.420795
AAACCTGCAAGCCCAC-1               1581        3441.0       8.137170
AAACCTGCACAGACTT-1               1922        4640.0       7.650862

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [117]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 10000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 40208


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 40092
Number of cells beforeMT filter: 39793
Number of cells after MT filter: 38460


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [118]:
adata_final.write("data/Mouse_public_data/GSE205930/GSE205930_postQC.h5ad")

In [40]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE205930/GSE205930_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [41]:
adata_final.obs['sample'].value_counts()

sample
GSE205930_1    10214
GSE205930_4     8228
GSE205930_3     7469
GSE205930_2     7001
GSE205930_5     5548
Name: count, dtype: int64

## GSE155513 42372 40958

In [119]:
file_paths = [
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_ApoE_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_ApoE_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_ApoE_KO_22_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_0_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_26_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_ApoE_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_ApoE_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_ApoE_KO_22_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_0_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_26_wks_WD.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10",
    "Sample11",
    "Sample12",
    "Sample13",
    "Sample14"
    
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 1935
Number of cells in sample Sample2: 1575
Number of cells in sample Sample3: 2062
Number of cells in sample Sample4: 2498
Number of cells in sample Sample5: 1714
Number of cells in sample Sample6: 2742
Number of cells in sample Sample7: 3403
Number of cells in sample Sample8: 2932
Number of cells in sample Sample9: 3150
Number of cells in sample Sample10: 3426
Number of cells in sample Sample11: 3094
Number of cells in sample Sample12: 3513
Number of cells in sample Sample13: 4287
Number of cells in sample Sample14: 6041


In [120]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE155513/GSE155513_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [121]:
adata_final

AnnData object with n_obs × n_vars = 42372 × 15852
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [122]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 207 个包含 'mt' 的基因


In [123]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE155513/GSE155513_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [124]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                  n_genes_by_counts  total_counts  pct_counts_mt
TCTGAGAGTAAGTAGT               1280        2908.0       0.309491
TGATTTCAGCGTTTAC               1985        5160.0       1.279070
AGCGTCGAGATGGCGT               1441        3313.0       0.724419
GATTCAGCAACTGCTA               1463        3296.0       0.940534
CTCTAATCACATCTTT               1624        3802.0       0.789058

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Atp6', 'mt-Atp8', 'mt-Co1', 'mt-Co2', 'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3', 'mt-Nd4', 'mt-Nd4l', 'mt-Nd5', 'mt-Nd6']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [125]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 15000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 42372


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 42361
Number of cells beforeMT filter: 40962
Number of cells after MT filter: 40958


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [126]:
adata_final.write("data/Mouse_public_data/GSE155513/GSE155513_postQC.h5ad")

In [42]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE155513/GSE155513_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [43]:
adata_final.obs['sample'].value_counts()

sample
GSE155513_7     6041
GSE155513_5     3919
GSE155513_3     3511
GSE155513_8     3401
GSE155513_1     3094
GSE155513_13    2973
GSE155513_11    2793
GSE155513_9     2788
GSE155513_6     2741
GSE155513_2     2498
GSE155513_14    1980
GSE155513_10    1934
GSE155513_4     1713
GSE155513_12    1572
Name: count, dtype: int64

## GSE134557 2547 2160

In [46]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE134557/GSE134557.h5ad")

In [47]:
adata_final.obs['sample'].value_counts()

sample
apoend     851
apoehfd    762
dkohfd     668
dkond      266
Name: count, dtype: int64

In [128]:
adata_final

AnnData object with n_obs × n_vars = 2547 × 11326
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention'
    var: 'gene_id'

In [129]:
adata_final.var_names

Index(['Gnai3', 'Narf', 'Cav2', 'Klf6', 'Scmh1', 'Cox5a', 'Wnt9a', 'Fer',
       'Xpo6', 'Tfe3',
       ...
       'CU074400.1', 'AC170188.1', 'AC123738.1', 'AC154176.2', 'AC123935.1',
       'AC154767.3', 'AC154335.1', 'CT025566.1', 'AC126549.2', 'Nupl1'],
      dtype='object', name='gene_name', length=11326)

In [130]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 0 个以 'mt-' 开头的基因
找到 165 个包含 'mt' 的基因


In [131]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [132]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                         n_genes_by_counts  total_counts  pct_counts_mt
cell                                                                   
AAACGGGGTCAGATAA_apoend               3198   4619.825156            0.0
AAACGGGGTGTGACCC_apoend               3103   4687.382232            0.0
AAAGATGAGATATGCA_apoend               1554   3055.609811            0.0
AAAGATGAGCGTTTAC_apoend               3007   4737.938297            0.0
AAAGATGCATCCGCGA_apoend               1537   3045.752125            0.0

线粒体基因统计：
共标记了 0 个线粒体基因
这些基因是：[]


In [133]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 5000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 6] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 2547
Number of cells before counts filter: 2519
Number of cells beforeMT filter: 2160
Number of cells after MT filter: 2160


In [134]:
adata_final.write("data/Mouse_public_data/GSE134557/GSE134557_postQC.h5ad")

In [44]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE134557/GSE134557_postQC.h5ad")

In [45]:
adata_final.obs['sample'].value_counts()

sample
apoend     791
apoehfd    650
dkohfd     517
dkond      202
Name: count, dtype: int64

## GSE131776 27086 26203

In [50]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE131776/GSE131776.h5ad")

In [51]:
adata_final.obs['sample'].value_counts()

sample
GSE131776_3     2436
GSE131776_4     2371
GSE131776_16    2124
GSE131776_9     2091
GSE131776_8     2079
GSE131776_7     2008
GSE131776_6     1856
GSE131776_17    1707
GSE131776_5     1383
GSE131776_12    1371
GSE131776_11    1369
GSE131776_18    1351
GSE131776_1     1271
GSE131776_15    1125
GSE131776_2      758
GSE131776_13     667
GSE131776_10     590
GSE131776_14     529
Name: count, dtype: int64

In [136]:
adata_final

AnnData object with n_obs × n_vars = 27086 × 17976
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention'

In [137]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 208 个包含 'mt' 的基因


In [138]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [139]:
adata_final.var['mt']

Xkr4       False
Rp1        False
Sox17      False
Gm37323    False
Mrpl15     False
           ...  
mt-Nd4l     True
mt-Nd4      True
mt-Nd5      True
mt-Nd6      True
mt-Cytb     True
Name: mt, Length: 17976, dtype: bool

In [140]:
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

In [141]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=500)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=3500) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 6] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 27086
Number of cells before counts filter: 27086
Number of cells beforeMT filter: 27085
Number of cells after MT filter: 26203


In [142]:
adata_final.write("data/Mouse_public_data/GSE131776/GSE131776_postQC.h5ad")

In [52]:
adata_final =sc.read_h5ad("data/Mouse_public_data/GSE131776/GSE131776_postQC.h5ad")

In [53]:
adata_final.obs['sample'].value_counts()

sample
GSE131776_3     2362
GSE131776_4     2336
GSE131776_16    2049
GSE131776_8     2047
GSE131776_9     1963
GSE131776_7     1933
GSE131776_6     1833
GSE131776_17    1616
GSE131776_5     1350
GSE131776_12    1347
GSE131776_11    1325
GSE131776_18    1294
GSE131776_1     1219
GSE131776_15    1071
GSE131776_2      749
GSE131776_13     621
GSE131776_10     580
GSE131776_14     508
Name: count, dtype: int64

# AAA

## GSE264071 16770 15674

In [143]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE264071/GSE264071_KO.h5ad",
    "data/Mouse_public_data/exp_AS/GSE264071/GSE264071_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 9034
Number of cells in sample Sample2: 7736


In [144]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [145]:
adata_final

AnnData object with n_obs × n_vars = 16770 × 31253
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [146]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 253 个包含 'mt' 的基因


In [147]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [148]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                             n_genes_by_counts  total_counts  pct_counts_mt
AAATCCTGTAAACTGCGCCAACGATCT                652         963.0       1.349948
AAATCCTGTAAATAAATAACAGGCTCC               4169       16899.0       3.035683
AAATCCTGTAAATAAATAGGTTGGACA               1819        4375.0       3.771428
AAATCCTGTAAATAAATATCTCGGTTA                545        1204.0       1.162791
AAATCCTGTAACCAAAGTCAGGGAGGG                262         944.0      64.618645

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [149]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 16770


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 16516
Number of cells beforeMT filter: 16317
Number of cells after MT filter: 15674


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [150]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_postQC.h5ad")

In [54]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [55]:
adata_final.obs['sample'].value_counts()

sample
GSE264071_KO    8201
GSE264071_WT    7473
Name: count, dtype: int64

## GSE239620 92330 78250

In [151]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_AAA.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_AAACD45.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_AAD.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-I.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-II.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-IICD45.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-III.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-IIICD45.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_NA.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9" 
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 7471
Number of cells in sample Sample2: 13648
Number of cells in sample Sample3: 8349
Number of cells in sample Sample4: 9617
Number of cells in sample Sample5: 6794
Number of cells in sample Sample6: 11136
Number of cells in sample Sample7: 15856
Number of cells in sample Sample8: 10675
Number of cells in sample Sample9: 8784


In [152]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [153]:
adata_final

AnnData object with n_obs × n_vars = 92330 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [154]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [155]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [156]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCACAGCAGGAT-1               1878        5248.0       3.010671
AAACCCAGTAGGCAAC-1               1537        4473.0       2.481556
AAACCCAGTTTCTATC-1                463         947.0      24.076029
AAACCCATCTCACTCG-1               1136        2969.0       6.433143
AAACGAAGTATCCCAA-1               2601       12719.0       2.350814

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [157]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=4500) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 45000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 92330


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 84291
Number of cells beforeMT filter: 84205
Number of cells after MT filter: 78250


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [158]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_postQC.h5ad")

In [56]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [57]:
adata_final.obs['sample'].value_counts()

sample
GSE239620_2    13048
GSE239620_7    10959
GSE239620_6    10545
GSE239620_8    10320
GSE239620_4     8065
GSE239620_9     6846
GSE239620_3     6828
GSE239620_1     5918
GSE239620_5     5721
Name: count, dtype: int64

## GSE231306 27865 24328

In [159]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE231306/GSE231306_AAA.h5ad",
    "data/Mouse_public_data/exp_AS/GSE231306/GSE231306_CTR.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 14086
Number of cells in sample Sample2: 13779


In [160]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [161]:
adata_final

AnnData object with n_obs × n_vars = 27865 × 55450
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [162]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 277 个包含 'mt' 的基因


In [163]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [164]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGACCATTC-1               4355       15410.0       1.421155
AAACCCAAGCGCCCAT-1                932        1783.0       1.346046
AAACCCAAGCTCGACC-1               1672        3768.0       0.610403
AAACCCAAGTCGAAGC-1               1479        2806.0       5.951532
AAACCCACAATATCCG-1               3942       15313.0       2.220335

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [165]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 27865


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 26231
Number of cells beforeMT filter: 26223
Number of cells after MT filter: 24328


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [166]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_postQC.h5ad")

In [58]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [59]:
adata_final.obs['sample'].value_counts()

sample
GSE231306_1    12305
GSE231306_2    12023
Name: count, dtype: int64

## GSE253247 41315 40242

In [167]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_P1.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_P2.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_P3.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_V1.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_V2.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_V3.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
    
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()
    adata_list.append(adata_sample)
    

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample1: 2312


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample2: 9384


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample4: 8662


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample5: 4962
Number of cells in sample Sample6: 6414


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [168]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [169]:
adata_final

AnnData object with n_obs × n_vars = 41315 × 27998
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [170]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 231 个包含 'mt' 的基因


In [171]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [172]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGAACAATC-1               1848        5628.0       2.043355
AAACCTGAGAAGCCCA-1                996        2630.0       2.091255
AAACCTGAGACTAGGC-1               1746        4741.0       1.603037
AAACCTGTCCCGGATG-1               1838        7970.0       3.074028
AAACGGGAGAAGGGTA-1               1533        4802.0       1.207830

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [173]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=4500) # acc to author
sc.pp.filter_genes(adata_final, min_cells=5) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 15000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 41315


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 41153
Number of cells beforeMT filter: 40350
Number of cells after MT filter: 40242


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [174]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_postQC.h5ad")

In [60]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [61]:
adata_final.obs['sample'].value_counts()

sample
GSE253247_3    9378
GSE253247_2    9140
GSE253247_4    8477
GSE253247_7    6205
GSE253247_5    4807
GSE253247_1    2235
Name: count, dtype: int64

## GSE233625 41941 34100

In [175]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_STING.h5ad",
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_WTC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_WTC2000.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 13180
Number of cells in sample Sample2: 16053
Number of cells in sample Sample3: 12708


In [176]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [177]:
adata_final

AnnData object with n_obs × n_vars = 41941 × 32285
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [178]:
adata_final.var

""
gene_ids
ENSMUSG00000051951
ENSMUSG00000089699
ENSMUSG00000102331
ENSMUSG00000102343
ENSMUSG00000025900
...
ENSMUSG00000095523
ENSMUSG00000095475
ENSMUSG00000094855


In [179]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_merged.h5ad")
ensembl_id_df = pd.read_csv("mouse_gene_names_to_ensembl.csv")
# 核心修改：创建“基因ID→基因名”的映射字典（原代码是“基因名→基因ID”）
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
# 备份原始的基因ID
adata_final.var['original_ensembl_ids'] = adata_final.var_names

# 将AnnData的变量名（基因ID）转换为基因名
# 逻辑：如果基因ID在映射字典中，则替换为对应的基因名；否则保留原始ID
adata_final.var_names = [
    ensembl_to_gene[ensembl_id] if ensembl_id in ensembl_to_gene else ensembl_id 
    for ensembl_id in adata_final.var_names
]

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [180]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 265 个包含 'mt' 的基因


In [181]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [182]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGGATTTAG-1               3160       10056.0       4.494829
AAACCCACAACCGCCA-1               5811       33594.0       4.155504
AAACCCACAAGTATCC-1               2876       10363.0       6.272315
AAACCCACAATACAGA-1               3147       12023.0       8.142726
AAACCCACACAAGCAG-1               2386        7206.0       2.317513

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [183]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 41941


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 40981
Number of cells beforeMT filter: 36517
Number of cells after MT filter: 34100


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [184]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_postQC.h5ad")

In [62]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [63]:
adata_final.obs['sample'].value_counts()

sample
GSE233625_2    14657
GSE233625_1    10097
GSE233625_3     9346
Name: count, dtype: int64

## GSE237067 17389 14586

In [185]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_AAA7.h5ad",
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_AAA14.h5ad",
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_Sham.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 3496
Number of cells in sample Sample2: 10108
Number of cells in sample Sample3: 3785


In [186]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [187]:
adata_final

AnnData object with n_obs × n_vars = 17389 × 32285
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [188]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [189]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [190]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCACATAAGCAA-1                600        1202.0       0.332779
AAACCCACATGTCTAG-1                830        1830.0       6.830601
AAACCCACATGTTCGA-1               1092        2408.0       3.779070
AAACCCATCGAATGCT-1                325         596.0      20.973154
AAACGAAAGCGTGTTT-1                509         875.0      10.857142

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [191]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30284) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 17389
Number of cells before counts filter: 17177
Number of cells beforeMT filter: 17099
Number of cells after MT filter: 14586


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [192]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_postQC.h5ad")

In [64]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [65]:
adata_final.obs['sample'].value_counts()

sample
GSE237067_2    8187
GSE237067_1    3345
GSE237067_3    3054
Name: count, dtype: int64

## GSE191226 17274 15753

In [193]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE191226/GSE191226_D_A.h5ad",
    "data/Mouse_public_data/exp_AS/GSE191226/GSE191226_IN_A.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 9125
Number of cells in sample Sample2: 8149


In [194]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [195]:
adata_final

AnnData object with n_obs × n_vars = 17274 × 55450
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [196]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 277 个包含 'mt' 的基因


In [197]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [198]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGAAGTGTT-1               3582        8844.0       3.165988
AAACCCAAGGGAGATA-1               4558       18401.0       1.157546
AAACCCAAGGGTCACA-1               1828        4038.0       8.915304
AAACCCACAAATTAGG-1               1372        2654.0       3.165034
AAACCCACAAGTATCC-1                432         661.0      11.649016

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [199]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=500)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 200000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 25] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 17274


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before counts filter: 16039
Number of cells beforeMT filter: 16039
Number of cells after MT filter: 15753


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [200]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_postQC.h5ad")

In [66]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [67]:
adata_final.obs['sample'].value_counts()

sample
GSE191226_1    8278
GSE191226_2    7475
Name: count, dtype: int64

## GSE164678 5244 4541

In [201]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE164678/GSE164678_aaa.h5ad",
    "data/Mouse_public_data/exp_AS/GSE164678/GSE164678_sham.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 1922
Number of cells in sample Sample2: 3322


In [202]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [203]:
adata_final

AnnData object with n_obs × n_vars = 5244 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [204]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [205]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_merged.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [206]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAGTGGTCTTA-1               2205        5311.0       1.581623
AAACCCATCGGTCTAA-1               4367       23074.0       3.042385
AAACGAAGTATGGGAC-1                763        1308.0       0.611621
AAACGAAGTCGTGTTA-1               2325        7748.0      30.562725
AAACGCTAGATGACCG-1                930        2069.0       2.126631

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [207]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 200000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 5244
Number of cells before counts filter: 5113
Number of cells beforeMT filter: 5113
Number of cells after MT filter: 4541


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [208]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_postQC.h5ad")

In [68]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_postQC.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [69]:
adata_final.obs['sample'].value_counts()

sample
GSE164678_2    2886
GSE164678_1    1655
Name: count, dtype: int64

## GSE141031 13056 13019

In [74]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE141031/GSE141031_Apoe_0M_1M_2M_4M.h5ad",
    "data/Mouse_public_data/exp_AS/GSE141031/GSE141031_dko_0M_1M_2M_4M.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 6797
Number of cells in sample Sample2: 6259


In [75]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged.h5ad")

In [76]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged.h5ad")

In [77]:
adata_final.obs['sample'].value_counts()

sample
GSE141031_apoe3    2659
GSE141031_dko3     2253
GSE141031_dko4     1577
GSE141031_apoe2    1488
GSE141031_apoe1    1415
GSE141031_dko2     1302
GSE141031_apoe4    1235
GSE141031_dko1     1127
Name: count, dtype: int64

In [78]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged.h5ad")
ensembl_id_df = pd.read_csv("mouse_gene_names_to_ensembl.csv")
# 核心修改：创建“基因ID→基因名”的映射字典（原代码是“基因名→基因ID”）
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
# 备份原始的基因ID
adata_final.var['original_ensembl_ids'] = adata_final.var_names

# 将AnnData的变量名（基因ID）转换为基因名
# 逻辑：如果基因ID在映射字典中，则替换为对应的基因名；否则保留原始ID
adata_final.var_names = [
    ensembl_to_gene[ensembl_id] if ensembl_id in ensembl_to_gene else ensembl_id 
    for ensembl_id in adata_final.var_names
]

In [79]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 190 个包含 'mt' 的基因


In [80]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [81]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                          n_genes_by_counts  total_counts  pct_counts_mt
cell                                                                    
AAACCTGCATGCATGT_apoe_0m               2951   4248.938212       0.912954
AAACCTGGTACCGAGA_apoe_0m               1716   2964.945262       1.395850
AAACCTGGTATATCCG_apoe_0m               3966   5405.915428       0.715048
AAACCTGGTCTAACGT_apoe_0m               3445   5066.894762       0.808658
AAACCTGGTCTTCAAG_apoe_0m               3341   4884.659199       0.807413

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Rnr1', 'mt-Rnr2', 'mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb', 'mt-Tp']


In [82]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 8000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 3] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 13056
Number of cells before counts filter: 13019
Number of cells beforeMT filter: 13019
Number of cells after MT filter: 13019


In [83]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_postQC.h5ad")

In [84]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_postQC.h5ad")

In [85]:
adata_final.obs['sample'].value_counts()

sample
GSE141031_apoe3    2657
GSE141031_dko3     2251
GSE141031_dko4     1567
GSE141031_apoe2    1485
GSE141031_apoe1    1412
GSE141031_dko2     1297
GSE141031_apoe4    1234
GSE141031_dko1     1116
Name: count, dtype: int64

# merged

In [86]:
file_paths = [
    "data/Mouse_public_data/GSE239591/GSE239591_postQC.h5ad",
    "data/Mouse_public_data/GSE269449/GSE269449_postQC.h5ad",
    "data/Mouse_public_data/GSE298574/GSE298574_postQC.h5ad",
    "data/Mouse_public_data/GSE274572/GSE274572_postQC.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_postQC.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_postQC.h5ad",
    "data/Mouse_public_data/GSE284253/GSE284253_postQC.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_postQC.h5ad",
    "data/Mouse_public_data/GSE211216/GSE211216_postQC.h5ad",
    "data/Mouse_public_data/GSE245373/GSE245373_postQC_mt.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_postQC.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_postQC.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_postQC.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_postQC.h5ad",
    "data/Mouse_public_data/GSE134557/GSE134557_postQC.h5ad",
    "data/Mouse_public_data/GSE131776/GSE131776_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE141031/GSE141031_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE164678/GSE164678_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE191226/GSE191226_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE231306/GSE231306_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_postQC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE264071/GSE264071_postQC.h5ad"
]

datasets = [
    "Dataset1",
    "Dataset2",
    "Dataset3",
    "Dataset4",
    "Dataset5",
    "Dataset6",
    "Dataset7",
    "Dataset8",
    "Dataset9",
    "Dataset10",
    "Dataset11",
    "Dataset12",
    "Dataset13",
    "Dataset14",
    "Dataset15",
    "Dataset16",
    "Dataset17",
    "Dataset18",
    "Dataset19",
    "Dataset20",
    "Dataset21",
    "Dataset22",
    "Dataset23",
    "Dataset24",
    "Dataset25"
]
# 读取所有H5AD文件
adata_list = []
for path, datasets in zip(file_paths, datasets):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(datasets, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {datasets} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset1: 27094


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset2: 10158


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset23: 78250


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset24: 40242
Number of cells in sample Dataset25: 15674


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [87]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [88]:
adata_final.write_h5ad("output_data/public_data/Mouse_all/Mouse_merged.h5ad")

In [89]:
adata_final

AnnData object with n_obs × n_vars = 583018 × 57983
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'

In [222]:
#####################################37个基因都来自GSE274572 GSE210406#####################################

In [223]:
adata= sc.read_h5ad("output_data/public_data/Mouse_AS/Mouse_merged_mt.h5ad")

KeyboardInterrupt: 

In [322]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 13 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 499 个包含 'mt' 的基因


In [323]:
mt_genes_lower

['mt-Atp6',
 'mt-Atp8',
 'mt-Co1',
 'mt-Co2',
 'mt-Co3',
 'mt-Cytb',
 'mt-Nd1',
 'mt-Nd2',
 'mt-Nd3',
 'mt-Nd4',
 'mt-Nd4l',
 'mt-Nd5',
 'mt-Nd6',
 'mt-Rnr1',
 'mt-Rnr2',
 'mt-Ta',
 'mt-Tc',
 'mt-Td',
 'mt-Te',
 'mt-Tf',
 'mt-Tg',
 'mt-Th',
 'mt-Ti',
 'mt-Tk',
 'mt-Tl1',
 'mt-Tl2',
 'mt-Tm',
 'mt-Tn',
 'mt-Tp',
 'mt-Tq',
 'mt-Tr',
 'mt-Ts1',
 'mt-Ts2',
 'mt-Tt',
 'mt-Tv',
 'mt-Tw',
 'mt-Ty']